# M06. 조건부 필터링

> 📌 **언제 필요한가**  
> 특정 조건을 만족하는 행만 보고 싶을 때.  
> 예: "서울 지역만", "2023년 데이터만", "졸업자 1000명 이상만"

## 이 모듈에서 배울 것

- 단일 조건 필터링 (`df[조건]`)
- 여러 조건 결합 (`&`, `|`)
- 문자열 조건 (`str.contains`, `isin`)
- `query()` 메서드 (가독성)

---


## 📥 데이터 준비

> 이 모듈은 아래 파일이 필요해요. **저장소에는 동봉돼 있지 않으니** 먼저 받아서 두세요.
> 받는 곳 링크를 누르면 바로 받으러 갈 수 있어요.

- `한국교육개발원_시도 시군구별 졸업자 진학자 진학률_20240401.csv` — 인코딩 `cp949` — [공공데이터포털에서 받기](https://www.data.go.kr/data/15053808/fileData.do)

두는 곳 — **로컬 Jupyter**: 이 노트북과 같은 폴더 / **Colab**: `/content/`에 업로드.  
컬럼 설명·함정 등 자세한 내용은 [`data/README.md`](data/README.md) 참고.

---

## 1. 단일 조건


In [ ]:
import pandas as pd

graduate = pd.read_csv('data/한국교육개발원 시도 시군구별 졸업자 진학자 진학률_20240401.csv', encoding='cp949')
print(f"전체: {len(graduate)}행")
graduate.head()


In [ ]:
# 졸업자 1000명 이상만
big = graduate[graduate['졸업자'] >= 1000]
print(f"필터 후: {len(big)}행")
big.head()


## 2. 여러 조건 결합

> ⚠️ pandas에서는 `and`/`or` 대신 `&`/`|` 사용. 그리고 각 조건은 괄호로 감싸기.


In [ ]:
# 서울 + 2024년
filtered = graduate[(graduate['시도'] == '서울') & (graduate['연도'] == 2024)]
print(f"서울 2024: {len(filtered)}행")
filtered.head()

In [ ]:
# OR 조건: 서울 또는 부산
filtered2 = graduate[(graduate['시도'] == '서울') | (graduate['시도'] == '부산')]
print(f"서울/부산: {len(filtered2)}행")

## 3. `isin` — 여러 값 중 하나


In [ ]:
# 광역시만 (or 조건 여러 개를 깔끔하게)
metros = ['부산', '대구', '인천', '광주', '대전', '울산']
metro_data = graduate[graduate['시도'].isin(metros)]
print(f"광역시: {len(metro_data)}행")
metro_data['시도'].value_counts()

## 4. 문자열 조건 — `str.contains`


In [ ]:
# '남'이 들어간 시도 (충남, 전남, 경남)
special = graduate[graduate['시도'].str.contains('남')]
print(f"'남' 포함: {len(special)}행")
special['시도'].unique()

## 5. `query()` — 가독성 좋은 방법


In [ ]:
# 같은 필터링을 query로
result = graduate.query('시도 == "서울" and 연도 == 2024')
print(f"shape: {result.shape}")
result.head()

In [ ]:
# 더 복잡한 조건
result2 = graduate.query('졸업자 > 1000 and 진학자 / 졸업자 > 0.8')
result2.head()


## 6. 컬럼 + 행 동시 필터


In [ ]:
# 서울 2024 데이터의 졸업자/진학자만
sub = graduate.loc[
    (graduate['시도'] == '서울') & (graduate['연도'] == 2024),
    ['시군구', '졸업자', '진학자']
]
sub.head()

## 7. 본인 데이터에 적용해보기 ✏️


In [ ]:
# 본인 데이터 필터링
# 
# # 단일 조건
# my_filter = my_df[my_df['컬럼'] > 100]
# 
# # 여러 조건
# my_filter = my_df[(my_df['컬럼1'] > 100) & (my_df['컬럼2'] == '값')]
# 
# # isin
# my_filter = my_df[my_df['지역'].isin(['서울', '부산'])]


## 8. ⚠️ 함정 / 주의사항

### 8.1 `and`/`or` 쓰면 에러
```python
df[df['A'] > 0 and df['B'] > 0]  # ❌ ValueError
df[(df['A'] > 0) & (df['B'] > 0)]  # ✅
```

### 8.2 괄호 깜빡
`&`는 우선순위가 `>`보다 높아서 괄호 필수.
```python
df[df['A'] > 0 & df['B'] > 0]    # ❌ 잘못 해석됨
df[(df['A'] > 0) & (df['B'] > 0)]  # ✅
```

### 8.3 결측치는 어떤 비교에도 False
`df['A'] > 0`에서 NaN인 행은 빠짐. 의도와 다를 수 있음.  
**해결**: `df['A'].fillna(0) > 0` 또는 `df.dropna()` 먼저.

### 8.4 카피 경고 (SettingWithCopyWarning)
필터링한 결과를 수정하면 경고 나올 수 있음.  
**해결**: `.copy()` 명시.
```python
sub = df[df['A'] > 0].copy()
sub['new'] = ...  # 경고 없음
```


## 9. 📚 더 알아보기

- `df.between(low, high)` — 범위 필터
- `df.where(cond)` — 조건 안 맞으면 NaN
- `df.mask(cond)` — where의 반대
- `df.filter()` — 컬럼/인덱스 이름 기반
